# Chapter 6: Hedge Portfolio Analysis Module

This chapter covers hedge portfolio analysis:
- Portfolio construction
- Correlation analysis
- Hedging strategy optimization
- Risk-return trade-off

## 1. Setup and Imports

In [ ]:
import datetime
import os
import sys
import warnings
from itertools import combinations

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.optimize import minimize

# Add project root to path
project_root = os.path.dirname(os.path.abspath('__file__'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import PERFORMANCE_CONFIG, setup_logging
from utils import DataLoader, PerformanceAnalyzer

logger = setup_logging('hedge_portfolio')

## 2. Portfolio Performance Calculator

In [ ]:
class PortfolioPerformance:
    """Portfolio performance calculation class"""
    
    def __init__(self):
        self.analyzer = PerformanceAnalyzer()
    
    def calculate_portfolio_returns(self, nav_data: pd.DataFrame, 
                                    weights: list) -> pd.Series:
        """Calculate portfolio returns
        
        Args:
            nav_data: DataFrame with NAV data (codes as columns)
            weights: Portfolio weights
            
        Returns:
            Portfolio returns series
        """
        returns = nav_data.pct_change().dropna()
        portfolio_returns = (returns * weights).sum(axis=1)
        return portfolio_returns
    
    def calculate_portfolio_nav(self, nav_data: pd.DataFrame,
                                weights: list) -> pd.Series:
        """Calculate portfolio NAV
        
        Args:
            nav_data: DataFrame with NAV data
            weights: Portfolio weights
            
        Returns:
            Portfolio NAV series
        """
        portfolio_nav = (nav_data * weights).sum(axis=1)
        return portfolio_nav
    
    def get_portfolio_metrics(self, nav_data: pd.DataFrame,
                              weights: list) -> dict:
        """Calculate all portfolio metrics
        
        Args:
            nav_data: DataFrame with NAV data
            weights: Portfolio weights
            
        Returns:
            Metrics dictionary
        """
        portfolio_nav = self.calculate_portfolio_nav(nav_data, weights)
        portfolio_returns = portfolio_nav.pct_change().dropna()
        
        return {
            'annual_return': self.analyzer.calculate_returns(portfolio_nav),
            'volatility': self.analyzer.calculate_volatility(portfolio_returns),
            'max_drawdown': self.analyzer.calculate_max_drawdown(portfolio_nav),
            'sharpe_ratio': self.analyzer.calculate_sharpe_ratio(portfolio_returns),
            'var_95': self.analyzer.calculate_var(portfolio_returns)
        }

# Initialize portfolio performance calculator
portfolio_perf = PortfolioPerformance()
print("Portfolio performance calculator initialized")

## 3. Hedge Portfolio Analyzer

In [ ]:
class HedgePortfolioAnalyzer:
    """Hedge portfolio analysis class"""
    
    def __init__(self, data_loader: DataLoader = None):
        """Initialize hedge portfolio analyzer
        
        Args:
            data_loader: DataLoader instance
        """
        self.data_loader = data_loader or DataLoader()
        self.analyzer = PerformanceAnalyzer()
        self.portfolio_perf = PortfolioPerformance()
    
    def get_correlation_matrix(self, returns_data: pd.DataFrame) -> pd.DataFrame:
        """Calculate correlation matrix
        
        Args:
            returns_data: DataFrame with returns data
            
        Returns:
            Correlation matrix DataFrame
        """
        return returns_data.corr()
    
    def find_low_correlation_pairs(self, returns_data: pd.DataFrame,
                                   threshold: float = 0.3) -> list:
        """Find pairs with low correlation
        
        Args:
            returns_data: DataFrame with returns data
            threshold: Correlation threshold
            
        Returns:
            List of low correlation pairs
        """
        corr_matrix = self.get_correlation_matrix(returns_data)
        
        low_corr_pairs = []
        codes = corr_matrix.columns.tolist()
        
        for i in range(len(codes)):
            for j in range(i + 1, len(codes)):
                corr = corr_matrix.iloc[i, j]
                if abs(corr) < threshold:
                    low_corr_pairs.append({
                        'code1': codes[i],
                        'code2': codes[j],
                        'correlation': corr
                    })
        
        return sorted(low_corr_pairs, key=lambda x: abs(x['correlation']))
    
    def optimize_hedge_weights(self, target_returns: pd.Series,
                               hedge_returns: pd.DataFrame) -> list:
        """Optimize hedge portfolio weights
        
        Args:
            target_returns: Target asset returns
            hedge_returns: Hedge asset returns DataFrame
            
        Returns:
            Optimized weights list
        """
        n_assets = hedge_returns.shape[1]
        
        def objective(weights):
            portfolio_return = (hedge_returns * weights).sum(axis=1)
            hedge_return = target_returns - portfolio_return
            return self.analyzer.calculate_volatility(hedge_return)
        
        # Constraints
        constraints = [
            {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}  # Weights sum to 1
        ]
        
        # Bounds (0 to 1 for each weight)
        bounds = [(0, 1) for _ in range(n_assets)]
        
        # Initial guess (equal weights)
        initial_weights = np.array([1/n_assets] * n_assets)
        
        result = minimize(
            objective,
            x0=initial_weights,
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )
        
        if result.success:
            return result.x.tolist()
        return initial_weights.tolist()
    
    def find_best_hedge_portfolios(self, target_code: str,
                                    candidate_codes: list,
                                    start_date: str,
                                    n_assets: int = 2,
                                    top_n: int = 10) -> list:
        """Find best hedge portfolios
        
        Args:
            target_code: Target ETF code
            candidate_codes: Candidate ETF codes
            start_date: Start date
            n_assets: Number of assets in hedge
            top_n: Number of top portfolios to return
            
        Returns:
            List of best hedge portfolios
        """
        # Get NAV data
        all_codes = [target_code] + candidate_codes
        nav_data = self.data_loader.get_etf_nav_data(all_codes, start_date)
        
        if nav_data.empty:
            return []
        
        # Pivot to wide format
        nav_pivot = nav_data.pivot(index='DT', columns='TRADE_CODE', values='NAV_ADJ')
        nav_pivot = nav_pivot.dropna()
        
        if nav_pivot.empty:
            return []
        
        # Calculate returns
        returns = nav_pivot.pct_change().dropna()
        target_returns = returns[target_code]
        
        # Find best combinations
        portfolios = []
        
        for codes in combinations(candidate_codes, n_assets - 1):
            codes = list(codes)
            hedge_returns = returns[codes]
            
            try:
                weights = self.optimize_hedge_weights(target_returns, hedge_returns)
                
                portfolio_return = (hedge_returns * weights).sum(axis=1)
                hedge_result = target_returns - portfolio_return
                
                portfolios.append({
                    'codes': codes,
                    'weights': weights,
                    'hedge_volatility': self.analyzer.calculate_volatility(hedge_result),
                    'correlation': hedge_result.corr(target_returns)
                })
                
            except Exception as e:
                continue
        
        # Sort by hedge volatility (lower is better)
        portfolios.sort(key=lambda x: x['hedge_volatility'])
        
        return portfolios[:top_n]

# Initialize hedge portfolio analyzer
hedge_analyzer = HedgePortfolioAnalyzer()
print("Hedge portfolio analyzer initialized")

## 4. Correlation Analysis

In [ ]:
def analyze_etf_correlations(nav_data: pd.DataFrame, 
                             etf_categories: pd.DataFrame) -> pd.DataFrame:
    """Analyze correlations between ETF categories
    
    Args:
        nav_data: NAV data in wide format
        etf_categories: ETF category mapping
        
    Returns:
        Correlation analysis DataFrame
    """
    returns = nav_data.pct_change().dropna()
    corr_matrix = returns.corr()
    
    # Map codes to categories
    category_map = dict(zip(etf_categories['TRADE_CODE'], 
                            etf_categories['market']))
    
    # Calculate category-level correlations
    results = []
    
    for i, code1 in enumerate(corr_matrix.columns):
        for j, code2 in enumerate(corr_matrix.columns):
            if i < j:
                results.append({
                    'code1': code1,
                    'code2': code2,
                    'category1': category_map.get(code1, 'unknown'),
                    'category2': category_map.get(code2, 'unknown'),
                    'correlation': corr_matrix.loc[code1, code2]
                })
    
    return pd.DataFrame(results)

def get_rolling_correlation(returns1: pd.Series, returns2: pd.Series,
                           window: int = 20) -> pd.Series:
    """Calculate rolling correlation
    
    Args:
        returns1: First returns series
        returns2: Second returns series
        window: Rolling window
        
    Returns:
        Rolling correlation series
    """
    return returns1.rolling(window=window).corr(returns2)

print("Correlation analysis functions defined")

## 5. Portfolio Optimization

In [ ]:
def optimize_portfolio(returns_data: pd.DataFrame,
                      target_return: float = None,
                      max_weight: float = 0.4) -> dict:
    """Optimize portfolio weights (mean-variance)
    
    Args:
        returns_data: DataFrame with returns data
        target_return: Target return (optional)
        max_weight: Maximum weight per asset
        
    Returns:
        Optimization results dictionary
    """
    n_assets = returns_data.shape[1]
    
    # Calculate expected returns and covariance
    expected_returns = returns_data.mean() * 252
    cov_matrix = returns_data.cov() * 252
    
    def portfolio_volatility(weights):
        return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    
    def portfolio_return(weights):
        return np.dot(weights, expected_returns)
    
    # Constraints
    constraints = [
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}  # Weights sum to 1
    ]
    
    if target_return:
        constraints.append({
            'type': 'eq',
            'fun': lambda x: portfolio_return(x) - target_return
        })
    
    # Bounds
    bounds = [(0, max_weight) for _ in range(n_assets)]
    
    # Initial guess
    initial_weights = np.array([1/n_assets] * n_assets)
    
    # Optimize
    result = minimize(
        portfolio_volatility,
        x0=initial_weights,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    
    if result.success:
        return {
            'weights': dict(zip(returns_data.columns, result.x)),
            'expected_return': portfolio_return(result.x),
            'volatility': result.fun,
            'sharpe': (portfolio_return(result.x) - 0.02) / result.fun
        }
    
    return {}

def calculate_efficient_frontier(returns_data: pd.DataFrame,
                                  n_points: int = 20) -> pd.DataFrame:
    """Calculate efficient frontier points
    
    Args:
        returns_data: DataFrame with returns data
        n_points: Number of points on frontier
        
    Returns:
        DataFrame with frontier points
    """
    expected_returns = returns_data.mean() * 252
    
    min_return = expected_returns.min()
    max_return = expected_returns.max()
    
    target_returns = np.linspace(min_return, max_return, n_points)
    
    frontier_points = []
    
    for target in target_returns:
        result = optimize_portfolio(returns_data, target_return=target)
        if result:
            frontier_points.append({
                'target_return': target,
                'expected_return': result['expected_return'],
                'volatility': result['volatility'],
                'sharpe': result['sharpe']
            })
    
    return pd.DataFrame(frontier_points)

print("Portfolio optimization functions defined")

## 6. 3+3 Hedge Strategy

In [ ]:
def build_3plus3_hedge_strategy(stock_etf: str, bond_etf: str,
                                 nav_data: pd.DataFrame,
                                 stock_weight: float = 0.5) -> dict:
    """Build 3+3 hedge strategy (3 stock + 3 bond)
    
    Args:
        stock_etf: Stock ETF code
        bond_etf: Bond ETF code
        nav_data: NAV data DataFrame
        stock_weight: Weight for stock ETF
        
    Returns:
        Strategy analysis results
    """
    # Filter data
    strategy_nav = nav_data[[stock_etf, bond_etf]].dropna()
    
    if strategy_nav.empty:
        return {}
    
    # Calculate portfolio NAV
    bond_weight = 1 - stock_weight
    weights = [stock_weight, bond_weight]
    
    portfolio_nav = (strategy_nav * weights).sum(axis=1)
    portfolio_returns = portfolio_nav.pct_change().dropna()
    
    # Calculate individual metrics
    stock_returns = strategy_nav[stock_etf].pct_change().dropna()
    bond_returns = strategy_nav[bond_etf].pct_change().dropna()
    
    analyzer = PerformanceAnalyzer()
    
    results = {
        'stock_etf': stock_etf,
        'bond_etf': bond_etf,
        'weights': {'stock': stock_weight, 'bond': bond_weight},
        'portfolio': {
            'annual_return': analyzer.calculate_returns(portfolio_nav),
            'volatility': analyzer.calculate_volatility(portfolio_returns),
            'max_drawdown': analyzer.calculate_max_drawdown(portfolio_nav),
            'sharpe_ratio': analyzer.calculate_sharpe_ratio(portfolio_returns)
        },
        'stock_only': {
            'annual_return': analyzer.calculate_returns(strategy_nav[stock_etf]),
            'volatility': analyzer.calculate_volatility(stock_returns),
            'max_drawdown': analyzer.calculate_max_drawdown(strategy_nav[stock_etf])
        },
        'bond_only': {
            'annual_return': analyzer.calculate_returns(strategy_nav[bond_etf]),
            'volatility': analyzer.calculate_volatility(bond_returns),
            'max_drawdown': analyzer.calculate_max_drawdown(strategy_nav[bond_etf])
        },
        'correlation': stock_returns.corr(bond_returns)
    }
    
    return results

def compare_hedge_ratios(stock_etf: str, bond_etf: str,
                         nav_data: pd.DataFrame) -> pd.DataFrame:
    """Compare different hedge ratios
    
    Args:
        stock_etf: Stock ETF code
        bond_etf: Bond ETF code
        nav_data: NAV data
        
    Returns:
        DataFrame with comparison results
    """
    ratios = [(0.7, 0.3), (0.6, 0.4), (0.5, 0.5), (0.4, 0.6), (0.3, 0.7)]
    
    results = []
    for stock_w, bond_w in ratios:
        strategy = build_3plus3_hedge_strategy(
            stock_etf, bond_etf, nav_data, stock_w
        )
        
        if strategy:
            results.append({
                'stock_weight': stock_w,
                'bond_weight': bond_w,
                **strategy['portfolio']
            })
    
    return pd.DataFrame(results)

print("3+3 hedge strategy functions defined")

## 7. Visualization

In [ ]:
def plot_correlation_heatmap(corr_matrix: pd.DataFrame, title: str = 'Correlation Heatmap'):
    """Plot correlation heatmap
    
    Args:
        corr_matrix: Correlation matrix
        title: Plot title
        
    Returns:
        Plotly Figure object
    """
    fig = go.Figure(data=go.Heatmap(
        z=corr_matrix.values,
        x=corr_matrix.columns,
        y=corr_matrix.index,
        colorscale='RdBu',
        zmin=-1,
        zmax=1,
        text=np.round(corr_matrix.values, 2),
        texttemplate='%{text}',
        textfont={'size': 10}
    ))
    
    fig.update_layout(
        title=title,
        template='plotly_white',
        width=800,
        height=800
    )
    
    return fig

def plot_efficient_frontier(frontier_df: pd.DataFrame,
                            individual_assets: pd.DataFrame = None):
    """Plot efficient frontier
    
    Args:
        frontier_df: DataFrame with frontier points
        individual_assets: Individual asset risk-return data
        
    Returns:
        Plotly Figure object
    """
    fig = go.Figure()
    
    # Plot frontier
    fig.add_trace(go.Scatter(
        x=frontier_df['volatility'] * 100,
        y=frontier_df['expected_return'] * 100,
        mode='lines',
        name='Efficient Frontier',
        line=dict(color='blue', width=2)
    ))
    
    # Plot individual assets
    if individual_assets is not None:
        fig.add_trace(go.Scatter(
            x=individual_assets['volatility'] * 100,
            y=individual_assets['return'] * 100,
            mode='markers',
            name='Individual Assets',
            marker=dict(size=10)
        ))
    
    fig.update_layout(
        title='Efficient Frontier',
        xaxis_title='Volatility (%)',
        yaxis_title='Expected Return (%)',
        template='plotly_white'
    )
    
    return fig

print("Visualization functions defined")

## 8. Summary

This chapter covered:
1. Portfolio performance calculation
2. Hedge portfolio analysis
3. Correlation analysis
4. Portfolio optimization (mean-variance)
5. 3+3 hedge strategy
6. Efficient frontier calculation
7. Visualization tools

### Key Classes
- `PortfolioPerformance`: Portfolio metrics calculation
- `HedgePortfolioAnalyzer`: Hedge strategy analysis

### Key Functions
- `optimize_portfolio`: Mean-variance optimization
- `find_best_hedge_portfolios`: Hedge portfolio selection
- `build_3plus3_hedge_strategy`: 3+3 strategy construction

### Best Practices
- Diversify across uncorrelated assets
- Consider risk-return trade-off
- Rebalance periodically
- Monitor correlation changes